In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import os
import pickle
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [2]:
X_train_feat = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\train_featured.csv')
X_test_feat = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\test_featured.csv')
y_train = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\y_train.csv')
y_test = pd.read_csv('D:\\Credit Risk Modelling\\Data\\featured\\y_test.csv')

### With Class Imbalance

In [3]:
model = XGBClassifier(random_state=42)
model.fit(X_train_feat, y_train)
y_pred = model.predict(X_test_feat)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     11390
           1       0.82      0.74      0.78      1108

    accuracy                           0.96     12498
   macro avg       0.90      0.86      0.88     12498
weighted avg       0.96      0.96      0.96     12498

[[11211   179]
 [  283   825]]


In [6]:
from scipy.stats import randint, uniform

y_train = y_train.squeeze()   # converts DataFrame column → Series
y_test = y_test.squeeze()     # converts DataFrame column → Series
# Calculate imbalance ratio
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    use_label_encoder=False
)

param_dist_imbalanced = {
    'n_estimators': randint(200, 800),
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5),
    'min_child_weight': randint(1, 10),
    'scale_pos_weight': [scale_pos_weight]   # 🔥 critical for imbalance
}

random_search_xgb_imbalanced = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_imbalanced,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search_xgb_imbalanced.fit(X_train_feat, y_train)

best_xgb_imbalanced = random_search_xgb_imbalanced.best_estimator_

print("Best Parameters:", random_search_xgb_imbalanced.best_params_)
print("Best CV AUC:", random_search_xgb_imbalanced.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\Credit Risk Modelling\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:11:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'colsample_bytree': np.float64(0.6488152939379115), 'gamma': np.float64(2.475884550556351), 'learning_rate': np.float64(0.02031655633456552), 'max_depth': 3, 'min_child_weight': 4, 'n_estimators': 761, 'scale_pos_weight': 10.755095641266855, 'subsample': np.float64(0.8650089137415928)}
Best CV AUC: 0.9870418919631131


In [7]:
from sklearn.metrics import roc_auc_score

y_proba_xgb = best_xgb_imbalanced.predict_proba(X_test_feat)[:, 1]
test_auc_xgb = roc_auc_score(y_test, y_proba_xgb)
print(classification_report(y_test, best_xgb_imbalanced.predict(X_test_feat)))
print("Test AUC (XGB):", test_auc_xgb)

              precision    recall  f1-score   support

           0       1.00      0.93      0.96     11390
           1       0.58      0.96      0.72      1108

    accuracy                           0.94     12498
   macro avg       0.79      0.95      0.84     12498
weighted avg       0.96      0.94      0.94     12498

Test AUC (XGB): 0.9877402909005618


### Without Class Imbalance

In [8]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_feat_sm, y_train_sm = smote.fit_resample(X_train_feat, y_train)

print("Before SMOTE:", y_train.value_counts())
print("After SMOTE:", y_train_sm.value_counts())

Before SMOTE: default
0    34298
1     3189
Name: count, dtype: int64
After SMOTE: default
0    34298
1    34298
Name: count, dtype: int64


In [10]:
model = XGBClassifier(random_state=42)
model.fit(X_train_feat_sm, y_train_sm)
y_pred = model.predict(X_test_feat)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98     11390
           1       0.75      0.81      0.78      1108

    accuracy                           0.96     12498
   macro avg       0.86      0.89      0.88     12498
weighted avg       0.96      0.96      0.96     12498

[[11089   301]
 [  212   896]]


In [11]:
param_dist_balanced = {
    'n_estimators': randint(200, 800),
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5),
    'min_child_weight': randint(1, 10)
}

random_search_xgb_balanced = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist_balanced,
    n_iter=50,
    scoring='roc_auc',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search_xgb_balanced.fit(X_train_feat_sm, y_train_sm)

best_xgb_balanced = random_search_xgb_balanced.best_estimator_

print("Best Parameters:", random_search_xgb_balanced.best_params_)
print("Best CV AUC:", random_search_xgb_balanced.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


d:\Credit Risk Modelling\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:16:26] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'colsample_bytree': np.float64(0.7001005442063456), 'gamma': np.float64(0.194173672147116), 'learning_rate': np.float64(0.10097965440196684), 'max_depth': 7, 'min_child_weight': 1, 'n_estimators': 463, 'subsample': np.float64(0.9079974212394444)}
Best CV AUC: 0.9984830147689209


In [12]:
from sklearn.metrics import roc_auc_score

y_proba_xgb = best_xgb_balanced.predict_proba(X_test_feat)[:, 1]
test_auc_xgb = roc_auc_score(y_test, y_proba_xgb)
print(classification_report(y_test, best_xgb_balanced.predict(X_test_feat)))
print("Test AUC (XGB):", test_auc_xgb)

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     11390
           1       0.78      0.80      0.79      1108

    accuracy                           0.96     12498
   macro avg       0.88      0.89      0.88     12498
weighted avg       0.96      0.96      0.96     12498

Test AUC (XGB): 0.9856501364487819


### Optuna

In [13]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer,f1_score

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 2),
        "scale_pos_weight": scale_pos_weight,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "random_state": 42,
        "n_jobs": -1
    }
        
    model_optuna = XGBClassifier(**params)
    f1_scorer = make_scorer(f1_score,average='macro')
    score = cross_val_score(model_optuna, X_train_feat_sm, y_train_sm, cv=5, scoring=f1_scorer).mean()

    return np.mean(score)
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

d:\Credit Risk Modelling\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-28 20:12:33,008] A new study created in memory with name: no-name-589a5f09-9e48-4293-9d98-cb6d5c92d218
[I 2026-02-28 20:12:36,202] Trial 0 finished with value: 0.9736321603108801 and parameters: {'n_estimators': 223, 'max_depth': 10, 'learning_rate': 0.16742735336948594, 'subsample': 0.8201362735387797, 'colsample_bytree': 0.7199839237300467, 'gamma': 1.1368563738783455, 'min_child_weight': 3, 'reg_alpha': 0.46397306704396246, 'reg_lambda': 0.8365302981637959}. Best is trial 0 with value: 0.9736321603108801.
[I 2026-02-28 20:12:41,323] Trial 1 finished with value: 0.9705944333679406 and parameters: {'n_estimators': 690, 'max_depth': 8, 'learning_rate': 0.1778293330290221, 'subsample': 0.6626721376990831, 'colsample_bytree': 0.

In [14]:
print("Best trial:")
trial = study.best_trial
print(' F1 Score: {}'.format(trial.value))
print('Parameters: ')
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

best_model_optuna = XGBClassifier(random_state=42, **trial.params)
best_model_optuna.fit(X_train_feat_sm, y_train_sm)
y_pred_optuna = best_model_optuna.predict(X_test_feat)
print(classification_report(y_test, y_pred_optuna))

Best trial:
 F1 Score: 0.9773104806648807
Parameters: 
    n_estimators: 348
    max_depth: 11
    learning_rate: 0.2169764692430895
    subsample: 0.7801636771985928
    colsample_bytree: 0.981747513218394
    gamma: 0.007067679810017446
    min_child_weight: 2
    reg_alpha: 0.03370198538741248
    reg_lambda: 0.010873089865655716
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     11390
           1       0.76      0.80      0.78      1108

    accuracy                           0.96     12498
   macro avg       0.87      0.89      0.88     12498
weighted avg       0.96      0.96      0.96     12498

